# EDA - Khám phá bộ dữ liệu UIT-VSFC

Notebook này gọi lại các hàm trong `src/eda.py`, nên kết quả khớp với báo cáo.

**Chọn kernel** là môi trường ảo `.venv` của dự án (góc trên bên phải VS Code).

In [ ]:
import os, sys
from pathlib import Path

# Đưa thư mục gốc dự án vào sys.path để `import src...` hoạt động dù notebook chạy từ thư mục nào.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src import config as C

## 1. Nạp dữ liệu
Tự đọc `data/raw/*.csv`; nếu chưa có sẽ tải từ Hugging Face (cần Internet).

In [ ]:
from src.data_loader import load_splits, check_sizes

splits = load_splits()
check_sizes(splits)
splits['train'].head(10)

## 2. Thống kê tổng quan

In [ ]:
from src.eda import build_summary

summary = build_summary(splits)
print('Kích thước     :', summary['sizes'])
print('Độ dài (số từ) :', summary['length_words'])
print('Tỉ lệ lớp lớn nhất / nhỏ nhất (train):', summary['imbalance_ratio_train'])
print('Câu trùng trong train        :', summary['duplicates_in_train'])
print('Câu test đã có trong train   :', summary['test_sentences_seen_in_train'])

for task in C.TASKS:
    counts = pd.DataFrame(summary['class_counts'][task], index=C.LABEL_VI[task])
    counts['train %'] = (100 * counts['train'] / counts['train'].sum()).round(1)
    display(counts)

## 3. Biểu đồ
Hàm `run_eda()` vẽ và lưu hình vào `results/figures/`; ở đây hiển thị lại.

In [ ]:
from src.eda import run_eda

run_eda()
for name in ['class_distribution_sentiment', 'class_distribution_topic', 'length_distribution', 'sentiment_by_topic']:
    display(Image(filename=str(C.FIGURES_DIR / f'{name}.png'), width=900))

## 4. Từ khóa nổi bật theo lớp (tập train)

In [ ]:
for task in C.TASKS:
    print(f'--- {task} ---')
    for label, words in summary['top_words'][task].items():
        print(f'{label:<18}', ', '.join(f'{w} ({c})' for w, c in words))

## 5. Ví dụ tiền xử lý

In [ ]:
from src.preprocess import clean_text, normalize_teencode, segment_words

for text in splits['train'][C.TEXT_COL].sample(5, random_state=C.RANDOM_STATE):
    cleaned = normalize_teencode(clean_text(text))
    print('gốc :', text)
    print('sạch:', cleaned)
    print('tách:', segment_words(cleaned))
    print()

### Nhận xét của bạn
_(Viết nhận xét về độ mất cân bằng lớp, độ dài câu, quan hệ chủ đề - cảm xúc, câu trùng giữa train và test...)_